In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field


In [12]:
###  Document Load
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()


### split
splitter = RecursiveCharacterTextSplitter(chunk_size= 1000, chunk_overlap=200)
docs = splitter.split_documents(docs)



### embedding and vector DB
embed_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

vector_store = InMemoryVectorStore.from_documents(
    documents= docs,
    embedding=embed_model
)

In [ ]:
class RagState(BaseModel):
    question: str = Field(description= "User Question")
    documents: list
    context:str = Field(description="Context data for user question")
    answer:str = Field(description="Final Answer")   ## this description is useful when we want structured o/p

##### Node  :    question -> retrive  ->  context  ->  generate  ->  end

In [ ]:
def retrieve_node(state:RagState) -> RagState:
    docs = vector_store.similarity_search(query=state.question)
    state.documents = docs
    return state



def create_context(state: RagState) -> RagState:
    context = ""
    for doc in state.documents:
        context += doc.page_content + "\n\n"

    state.context = context
    return state 


